# Visão Geral
O notebook implementa uma arquitetura Medalhão (Bronze → Silver → Gold) sobre o dataset _samples.nyctaxi.trips_, que contém 21.932 registros e 6 colunas originais (datas de pickup/dropoff, distância, valor da corrida e CEPs de origem/destino). O fluxo percorre coleta, limpeza, enriquecimento e modelagem analítica, culminando em 5 perguntas de negócio respondidas com SQL e visualizações Python para complementar o entendimento.

### Legenda — Variáveis e Tabelas do Pipeline

Esta seção descreve o significado completo de cada coluna e tabela utilizada no notebook, do dado bruto às agregações finais.

---

#### Tabela de origem: `samples.nyctaxi.trips`
Dataset público de corridas de táxi amarelo (Yellow Cab) da cidade de Nova York, disponibilizado pela Databricks como exemplo.

| Coluna | Tipo | Significado |
| --- | --- | --- |
| `tpep_pickup_datetime` | timestamp | Momento exato em que o passageiro entrou no táxi (início da corrida). *TPEP* significa *Taxi and Limousine Commission Passenger Enhancement Program*, o sistema de registradores eletrônicos instalados nos veículos. |
| `tpep_dropoff_datetime` | timestamp | Momento exato em que o passageiro saiu do táxi (fim da corrida). |
| `trip_distance` | double | Distância percorrida durante a corrida, medida em milhas, registrada pelo taxímetro do veículo. |
| `fare_amount` | double | Valor da tarifa cobrada do passageiro pelo taxímetro, em dólares americanos. Não inclui gorjetas, pedágios ou adicionais — apenas a tarifa base calculada por distância e tempo. |
| `pickup_zip` | int | Código postal (ZIP code) da zona onde o passageiro foi embarcado. Corresponde a áreas de Manhattan e arredores. |
| `dropoff_zip` | int | Código postal (ZIP code) da zona onde o passageiro foi desembarcado. |

---

#### Camada Bronze: `workspace.bronze.nyctaxi_trips`
Cópia fiel e bruta dos dados de origem, enriquecida com metadados de rastreabilidade. Nenhuma transformação de valor é aplicada — o objetivo é preservar o dado exatamente como chegou.

| Coluna | Tipo | Significado |
| --- | --- | --- |
| `tpep_pickup_datetime` | timestamp | (Mesma definição da origem.) Momento exato do início da corrida. |
| `tpep_dropoff_datetime` | timestamp | (Mesma definição da origem.) Momento exato do fim da corrida. |
| `trip_distance` | double | (Mesma definição da origem.) Distância percorrida em milhas. |
| `fare_amount` | double | (Mesma definição da origem.) Tarifa base em dólares. |
| `pickup_zip` | int | (Mesma definição da origem.) CEP do embarque. |
| `dropoff_zip` | int | (Mesma definição da origem.) CEP do desembarque. |
| `ingestion_timestamp` | timestamp | Carimbo de data/hora gerado automaticamente no momento em que o dado foi copiado para a Bronze. Permite saber quando cada carga ocorreu, essencial para auditoria e reprodução. |
| `source_table` | string | Nome completo da tabela de origem (`samples.nyctaxi.trips`). Permite rastrear de onde cada registro veio, útil quando múltiplas fontes alimentam a Bronze. |

---

#### Camada Silver: `workspace.silver.nyctaxi_trips`
Dado limpo, filtrado e enriquecido. Registros com distância inválida (≤ 0), tarifa inválida (≤ 0) ou com dropoff anterior ao pickup são removidos. Novas colunas derivadas são calculadas para suportar as análises de negócio.

| Coluna | Tipo | Significado |
| --- | --- | --- |
| `tpep_pickup_datetime` | timestamp | (Mesma definição.) Momento exato do início da corrida — agora garantidamente válido. |
| `tpep_dropoff_datetime` | timestamp | (Mesma definição.) Momento exato do fim da corrida — agora garantidamente posterior ao pickup. |
| `trip_distance` | double | Distância em milhas — agora garantidamente maior que zero. |
| `fare_amount` | double | Tarifa em dólares — agora garantidamente maior que zero. |
| `pickup_zip` | int | CEP do embarque, preservado da Bronze. |
| `dropoff_zip` | int | CEP do desembarque, preservado da Bronze. |
| `duration_minutes` | decimal(24,2) | **Derivada:** duração total da corrida em minutos, calculada como a diferença entre `tpep_dropoff_datetime` e `tpep_pickup_datetime` convertida para minutos e arredondada a duas casas decimais. |
| `pickup_hour` | int | **Derivada:** hora do dia (0 a 23) extraída de `tpep_pickup_datetime`. Permite agrupar corridas por faixa horária e identificar picos de demanda. |
| `pickup_dayofweek` | int | **Derivada:** dia da semana (1 = Domingo, 7 = Sábado) extraído de `tpep_pickup_datetime`. Permite comparar volume e receita entre dias da semana. |
| `ingestion_timestamp` | timestamp | Metadado de rastreabilidade, preservado da Bronze. |
| `source_table` | string | Metadado de origem, preservado da Bronze. |

---

#### Camada Gold: tabelas agregadas
As tabelas Gold contêm dados prontos para análise e visualização, pré-agregados em diferentes granularidades.

##### `workspace.gold.trips_by_hour`
Agrupa todas as corridas por hora do dia (0 a 23). Cada linha resume o comportamento da frota em uma determinada hora.

| Coluna | Tipo | Significado |
| --- | --- | --- |
| `pickup_hour` | int | Hora do dia (0 a 23) em que as corridas foram iniciadas. |
| `total_trips` | long | Número total de corridas iniciadas naquela hora. |
| `avg_fare` | double | Tarifa média (em dólares) das corridas daquela hora. |
| `avg_distance` | double | Distância média (em milhas) das corridas daquela hora. |
| `avg_duration` | decimal(25,2) | Duração média (em minutos) das corridas daquela hora. |
| `total_revenue` | double | Receita total (em dólares) somada de todas as corridas daquela hora. |

##### `workspace.gold.trips_by_dayofweek`
Agrupa todas as corridas por dia da semana (1 a 7). Inclui o nome do dia em português para facilitar a leitura.

| Coluna | Tipo | Significado |
| --- | --- | --- |
| `pickup_dayofweek` | int | Número do dia da semana (1 = Domingo, 7 = Sábado). |
| `day_name` | string | Nome do dia da semana em português (ex.: "Sexta", "Sábado"), derivado por uma expressão `CASE`. |
| `total_trips` | long | Número total de corridas iniciadas naquele dia da semana. |
| `avg_fare` | double | Tarifa média (em dólares) das corridas daquele dia. |
| `avg_distance` | double | Distância média (em milhas) das corridas daquele dia. |
| `total_revenue` | double | Receita total (em dólares) somada de todas as corridas daquele dia. |

##### `workspace.gold.top_routes`
Agrupa as corridas por par de CEP (origem → destino) e mantém apenas as 50 rotas mais frequentes.

| Coluna | Tipo | Significado |
| --- | --- | --- |
| `pickup_zip` | int | CEP onde o passageiro foi embarcado. |
| `dropoff_zip` | int | CEP onde o passageiro foi desembarcado. |
| `total_trips` | long | Número total de corridas que percorreram essa rota (par de CEPs). |
| `avg_fare` | double | Tarifa média (em dólares) dessa rota. |
| `avg_distance` | double | Distância média (em milhas) dessa rota. |
| `total_revenue` | double | Receita total (em dólares) acumulada nessa rota. |

---

## Fluxo de dependências

```
samples.nyctaxi.trips (origem)
        │
        ▼
workspace.bronze.nyctaxi_trips (cópia bruta + metadados)
        │
        ▼
workspace.silver.nyctaxi_trips (limpeza + colunas derivadas)
        │
        ├─▶ workspace.gold.trips_by_hour
        ├─▶ workspace.gold.trips_by_dayofweek
        └─▶ workspace.gold.top_routes
```

Cada camada consome exclusivamente a camada imediatamente anterior, seguindo o princípio do Medalhão: a Bronze nunca é alterada após a carga; a Silver filra e enriquece a Bronze; a Gold agrega a Silver.

# Explorando o Dataset

In [0]:
%sql
-- Ver a estrutura da tabela
DESCRIBE TABLE samples.nyctaxi.trips;

In [0]:
%sql
-- Ver as primeiras linhas
SELECT * FROM samples.nyctaxi.trips LIMIT 20;

In [0]:
%sql
-- Contar quantos registros existem
SELECT COUNT(*) AS total_corridas FROM samples.nyctaxi.trips;

### I. Usando o Catálogo WORKSPACE

Criar a Camada Bronze

In [0]:
%sql
SHOW CATALOGS;

# Criando os schemas Bronze, Silver e Gold

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

## I. Criando a Tabela Bronze (dado bruto)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.nyctaxi_trips
USING DELTA
AS
SELECT 
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  trip_distance,
  fare_amount,
  pickup_zip,
  dropoff_zip,
  current_timestamp() AS ingestion_timestamp,
  'samples.nyctaxi.trips' AS source_table
FROM samples.nyctaxi.trips;

#### (Verificações das Tabelas Bronze)

In [0]:
%sql
-- Estrutura da tabela Bronze 
DESCRIBE TABLE workspace.bronze.nyctaxi_trips;

In [0]:
%sql
-- Contagem (deve continuar 21932)
SELECT COUNT(*) AS total_bronze FROM workspace.bronze.nyctaxi_trips;

In [0]:
%sql
-- Ver as primeiras linhas (agora com os metadados novos)
SELECT * FROM workspace.bronze.nyctaxi_trips LIMIT 10;

### a) Análise de Qualidade de Dados

In [0]:
%sql
-- Completude – Valores nulos
SELECT 
  COUNT(*) AS total_registros,
  SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS nulos_pickup,
  SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS nulos_dropoff,
  SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS nulos_distance,
  SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) AS nulos_fare,
  SUM(CASE WHEN pickup_zip IS NULL THEN 1 ELSE 0 END) AS nulos_pickup_zip,
  SUM(CASE WHEN dropoff_zip IS NULL THEN 1 ELSE 0 END) AS nulos_dropoff_zip
FROM workspace.bronze.nyctaxi_trips;

In [0]:
%sql
-- Estatísticas básicas (média, mínimo, máximo, desvio)
SELECT 
  ROUND(AVG(trip_distance), 2) AS media_distancia,
  ROUND(MIN(trip_distance), 2) AS min_distancia,
  ROUND(MAX(trip_distance), 2) AS max_distancia,
  ROUND(AVG(fare_amount), 2) AS media_valor,
  ROUND(MIN(fare_amount), 2) AS min_valor,
  ROUND(MAX(fare_amount), 2) AS max_valor
FROM workspace.bronze.nyctaxi_trips;

### b) Outliers e valores suspeitos

In [0]:
%sql
-- Distância zero ou negativa
SELECT COUNT(*) AS distancia_zero_ou_negativa
FROM workspace.bronze.nyctaxi_trips
WHERE trip_distance <= 0;

In [0]:
%sql
-- Valor zero ou negativo
SELECT COUNT(*) AS valor_zero_ou_negativo
FROM workspace.bronze.nyctaxi_trips
WHERE fare_amount <= 0;

In [0]:
%sql
-- Corridas com duração negativa ou zero (dropoff antes do pickup)
SELECT COUNT(*) AS duracao_invalida
FROM workspace.bronze.nyctaxi_trips
WHERE tpep_dropoff_datetime <= tpep_pickup_datetime;

In [0]:
%sql
-- Valores extremos de distância e valor
SELECT 
  COUNT(*) AS distancias_muito_altas
FROM workspace.bronze.nyctaxi_trips
WHERE trip_distance > 100;

SELECT 
  COUNT(*) AS valores_muito_altos
FROM workspace.bronze.nyctaxi_trips
WHERE fare_amount > 500;

### c) Unicidade e distribuição de CEPs

In [0]:
%sql
-- Quantos CEPs de pickup e dropoff únicos existem
SELECT 
  COUNT(DISTINCT pickup_zip) AS ceps_pickup_unicos,
  COUNT(DISTINCT dropoff_zip) AS ceps_dropoff_unicos
FROM workspace.bronze.nyctaxi_trips;

## II. Criando a Tabela Silver (dado limpo)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.nyctaxi_trips
USING DELTA
AS
SELECT 
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  trip_distance,
  fare_amount,
  pickup_zip,
  dropoff_zip,
  -- Duração da corrida em minutos
  ROUND((UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60.0, 2) AS duration_minutes,
  -- Hora do dia (0-23)
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  -- Dia da semana (1 = Domingo, 7 = Sábado)
  DAYOFWEEK(tpep_pickup_datetime) AS pickup_dayofweek,
  ingestion_timestamp,
  source_table
FROM workspace.bronze.nyctaxi_trips
WHERE trip_distance > 0
  AND fare_amount > 0
  AND tpep_dropoff_datetime > tpep_pickup_datetime;

#### (Verificações da Tabela Silver)

In [0]:
%sql
-- Quantos registros sobraram
SELECT COUNT(*) AS total_silver FROM workspace.silver.nyctaxi_trips;

In [0]:
%sql
-- Conferindo a estrutura e uma amostra
DESCRIBE TABLE workspace.silver.nyctaxi_trips;

In [0]:
%sql
SELECT * FROM workspace.silver.nyctaxi_trips LIMIT 10;

## III. Camada Gold (dado pronto)

In [0]:
%sql
-- Resumo por hora do dia
CREATE OR REPLACE TABLE workspace.gold.trips_by_hour
USING DELTA
AS
SELECT 
  pickup_hour,
  COUNT(*) AS total_trips,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance,
  ROUND(AVG(duration_minutes), 2) AS avg_duration,
  ROUND(SUM(fare_amount), 2) AS total_revenue
FROM workspace.silver.nyctaxi_trips
GROUP BY pickup_hour
ORDER BY pickup_hour;

In [0]:
%sql
-- Resumo por dia da semana (day of week)
CREATE OR REPLACE TABLE workspace.gold.trips_by_dayofweek
USING DELTA
AS
SELECT 
  pickup_dayofweek,
  CASE pickup_dayofweek
    WHEN 1 THEN 'Domingo'
    WHEN 2 THEN 'Segunda'
    WHEN 3 THEN 'Terça'
    WHEN 4 THEN 'Quarta'
    WHEN 5 THEN 'Quinta'
    WHEN 6 THEN 'Sexta'
    WHEN 7 THEN 'Sábado'
  END AS day_name,
  COUNT(*) AS total_trips,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance,
  ROUND(SUM(fare_amount), 2) AS total_revenue
FROM workspace.silver.nyctaxi_trips
GROUP BY pickup_dayofweek
ORDER BY pickup_dayofweek;

In [0]:
%sql
-- Top rotas (pares de CEP)
CREATE OR REPLACE TABLE workspace.gold.top_routes
USING DELTA
AS
SELECT 
  pickup_zip,
  dropoff_zip,
  COUNT(*) AS total_trips,
  ROUND(AVG(fare_amount), 2) AS avg_fare,
  ROUND(AVG(trip_distance), 2) AS avg_distance,
  ROUND(SUM(fare_amount), 2) AS total_revenue
FROM workspace.silver.nyctaxi_trips
GROUP BY pickup_zip, dropoff_zip
ORDER BY total_trips DESC
LIMIT 50;

#### (Verificações das Tabelas Gold)

In [0]:
%sql
SELECT * FROM workspace.gold.trips_by_hour ORDER BY pickup_hour;

In [0]:
%sql
SELECT * FROM workspace.gold.trips_by_dayofweek;

In [0]:
%sql
SELECT * FROM workspace.gold.top_routes LIMIT 10;

## Pergunta 1
Qual é a distribuição do valor da corrida e da distância? Existem outliers significativos?

In [0]:
%sql
SELECT 
  ROUND(AVG(fare_amount), 2) AS media_valor,
  ROUND(MIN(fare_amount), 2) AS min_valor,
  ROUND(MAX(fare_amount), 2) AS max_valor,
  ROUND(PERCENTILE(fare_amount, 0.5), 2) AS mediana_valor,
  ROUND(AVG(trip_distance), 2) AS media_distancia,
  ROUND(MIN(trip_distance), 2) AS min_distancia,
  ROUND(MAX(trip_distance), 2) AS max_distancia,
  ROUND(PERCENTILE(trip_distance, 0.5), 2) AS mediana_distancia
FROM workspace.silver.nyctaxi_trips;

In [0]:
# Histogramas de "fare_amount" e "trip_distance" para visualizar a assimetria à direita e os outliers identificados na análise.

import matplotlib.pyplot as plt

# Carrega os dados limpos da camada Silver
df_silver = spark.table("workspace.silver.nyctaxi_trips")

# Converte colunas de interesse para pandas (apenas o necessário)
pdf_dist = df_silver.select("fare_amount", "trip_distance").toPandas()

# Cria figura com dois subplots lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma do valor da corrida
axes[0].hist(pdf_dist["fare_amount"], bins=60, color="#1f77b4", edgecolor="white", alpha=0.8)
axes[0].set_title("Distribuição do Valor da Corrida (fare_amount)")
axes[0].set_xlabel("Valor (US$)")
axes[0].set_ylabel("Frequência")
axes[0].axvline(pdf_dist["fare_amount"].median(), color="red", linestyle="--", label=f"Mediana: US$ {pdf_dist['fare_amount'].median():.2f}")
axes[0].axvline(pdf_dist["fare_amount"].mean(), color="orange", linestyle="--", label=f"Média: US$ {pdf_dist['fare_amount'].mean():.2f}")
axes[0].legend()

# Histograma da distância
axes[1].hist(pdf_dist["trip_distance"], bins=60, color="#2ca02c", edgecolor="white", alpha=0.8)
axes[1].set_title("Distribuição da Distância (trip_distance)")
axes[1].set_xlabel("Distância (milhas)")
axes[1].set_ylabel("Frequência")
axes[1].axvline(pdf_dist["trip_distance"].median(), color="red", linestyle="--", label=f"Mediana: {pdf_dist['trip_distance'].median():.2f} mi")
axes[1].axvline(pdf_dist["trip_distance"].mean(), color="orange", linestyle="--", label=f"Média: {pdf_dist['trip_distance'].mean():.2f} mi")
axes[1].legend()

plt.tight_layout()
plt.show()

### Resposta 1
A mediana é inferior à média, tanto em valor (med=9; méd=12.31) quanto em distância (med=1.7; méd=2.86), sugerindo que a maioria das corridas possui curta extensão e baixo valor, enquanto a média é elevada por um volume menor de corridas mais longas e caras.

A diferença entre a mediana e os valores máximos (respectivamente, 9 para 275 no valor; e 1.7 para 30.6 na distância) confirma a presença de outliers significativos que justificam uma assimetria.

Além disso, e felizmente, a remoção de valores zero ou negativos na etapa Silver resultou em dados com maior qualidade para a análise.

#### Conclusão
A maior parte das corridas de táxi neste dataset é de curta distância e baixo valor, o que é coerente com a dinâmica das ruas de Manhattan. Paralelamente, existem outliers de corridas longas e caras que devem ser considerados em análises de receita ou na definição de políticas de precificação, embora não representem o comportamento típico do sistema.


## Pergunta 2
Existe correlação clara entre distância e valor da corrida?

In [0]:
%sql
SELECT 
  ROUND(CORR(trip_distance, fare_amount), 4) AS correlacao_distancia_valor
FROM workspace.silver.nyctaxi_trips;

In [0]:
# Gráfico de dispersão para visualizar a correlação linear

import matplotlib.pyplot as plt
import numpy as np

# Carrega os dados da Silver (apenas as duas colunas necessárias)
df_silver = spark.table("workspace.silver.nyctaxi_trips")
pdf_corr = df_silver.select("trip_distance", "fare_amount").toPandas()

# Cria o scatter plot (gráfico de dispersão)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(pdf_corr["trip_distance"], pdf_corr["fare_amount"], alpha=0.3, s=10, color="#1f77b4")
ax.set_title("Correlação entre Distância e Valor da Corrida")
ax.set_xlabel("Distância (milhas)")
ax.set_ylabel("Valor (US$)")

# Linha de tendência linear
z = np.polyfit(pdf_corr["trip_distance"], pdf_corr["fare_amount"], 1)
p = np.poly1d(z)
x_line = np.linspace(pdf_corr["trip_distance"].min(), pdf_corr["trip_distance"].max(), 100)
ax.plot(x_line, p(x_line), color="red", linewidth=2, label="Tendência linear")

ax.legend()
plt.tight_layout()
plt.show()

### Resposta 2

Uma correlação de aproximadamente 0.95 é considerada muito forte e positiva, aproximando-se do valor máximo de 1, a correlação linear positiva perfeita ("-1" é a correlação linear negativa perfeita, e "0" é nulo). 

Isso indica que, neste conjunto de dados, a distância é o fator predominante na determinação do valor da corrida: quanto maior a distância, maior tende a ser o valor cobrado, seguindo uma tendência quase linear. Esse resultado é coerente com o funcionamento geral dos sistemas de táxi, que baseiam suas tarifas principalmente na distância e no tempo. A alta correlação confirma que o modelo de precificação apresenta um comportamento consistente nos dados.

#### Conclusão:

Sim, há uma correlação clara e significativa entre a distância e o valor da corrida, sendo que a distância explica a maior parte da variação do preço.

## Pergunta 3
Quais são os horários e dias da semana com maior volume e maior valor médio?

In [0]:
%sql
-- Horários de pico
SELECT * FROM workspace.gold.trips_by_hour
ORDER BY total_trips DESC
LIMIT 5;

In [0]:
%sql
-- Dias da semana
SELECT * FROM workspace.gold.trips_by_dayofweek
ORDER BY total_trips DESC;

In [0]:
# Dois gráficos de barras: volume de corridas por hora do dia e por dia da semana, usando as tabelas agregadas da Gold.

import matplotlib.pyplot as plt

# Carrega as tabelas Gold já agregadas
pdf_hour = spark.table("workspace.gold.trips_by_hour").toPandas()
pdf_day = spark.table("workspace.gold.trips_by_dayofweek").toPandas()

# Cria figura com dois subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Volume de corridas por hora ---
axes[0].bar(pdf_hour["pickup_hour"], pdf_hour["total_trips"], color="#1f77b4", edgecolor="white")
axes[0].set_title("Volume de Corridas por Hora do Dia")
axes[0].set_xlabel("Hora")
axes[0].set_ylabel("Total de Corridas")
axes[0].set_xticks(range(0, 24))

# Volume de corridas por dia da semana ---
day_order = ["Domingo", "Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado"]
pdf_day_sorted = pdf_day.set_index("day_name").loc[day_order].reset_index()
# Destaca a sexta-feira (dia de pico) em verde
colors = ["#2ca02c" if d == "Sexta" else "#1f77b4" for d in pdf_day_sorted["day_name"]]
axes[1].bar(pdf_day_sorted["day_name"], pdf_day_sorted["total_trips"], color=colors, edgecolor="white")
axes[1].set_title("Volume de Corridas por Dia da Semana")
axes[1].set_xlabel("Dia da Semana")
axes[1].set_ylabel("Total de Corridas")

plt.tight_layout()
plt.show()

### Resposta 3
Quanto aos **horários**, o pico de demanda ocorre no início da noite (18h–19h), típico da hora do rush de saída do trabalho. A partir das 20h o volume diminui gradualmente, mas o valor médio da corrida sobe um pouco (possivelmente por corridas mais longas ou menos concorrência).

Já sobre as **semanas**, Sexta-feira é o dia com maior volume de corridas e a maior receita total, e Terça-feira é o dia mais fraco. O valor médio da corrida varia pouco ao longo da semana (fica entre US$ 11,69 e US$ 12,79). Isso indica que a diferença de receita entre os dias é impulsionada principalmente pelo volume de corridas, e não por tarifas mais altas.

#### Conclusão
A demanda é claramente concentrada no final da tarde/início da noite e no final de semana (especialmente sexta). Esses padrões são úteis para planejamento de frota e estratégias de precificação dinâmica.

## Pergunta 4
Quais são os CEPs e rotas mais frequentes / rentáveis?

In [0]:
%sql
-- Rotas mais frequentes (por volume de corridas)
SELECT * FROM workspace.gold.top_routes
ORDER BY total_trips DESC
LIMIT 10;

In [0]:
%sql
-- Rotas mais rentáveis (por receita total)
SELECT * FROM workspace.gold.top_routes
ORDER BY total_revenue DESC
LIMIT 10;

### Resposta 4
A maioria das rotas mais populares consiste em trajetos curtos e intra-bairros (frequentemente com o mesmo CEP de origem e destino). Os CEPs 10021, 10023, 10028 e 10003 apresentam os maiores volumes de corridas e receita, correspondendo a regiões densas e de alto poder aquisitivo de Manhattan (Upper East Side e Upper West Side).

Neste conjunto de dados, não se observa a predominância de rotas longas, como trajetos para aeroportos ou inter-bairros, no ranking de receita. O alto volume de corridas curtas compensa o valor unitário mais baixo.

#### Conclusão:

O padrão predominante é de deslocamentos urbanos curtos dentro de Manhattan. As rotas com maior rentabilidade são aquelas com alto volume de corridas de baixo valor unitário, em vez de trajetos longos com valores individuais mais elevados.

## Pergunta 5 (Última)
Qual a duração média das corridas e como ela se relaciona com o valor?

In [0]:
%sql
SELECT 
  ROUND(AVG(duration_minutes), 2) AS media_duracao_minutos,
  ROUND(MIN(duration_minutes), 2) AS min_duracao,
  ROUND(MAX(duration_minutes), 2) AS max_duracao,
  ROUND(CORR(duration_minutes, fare_amount), 4) AS correlacao_duracao_valor
FROM workspace.silver.nyctaxi_trips;

### Resposta 5
a. A duração média de aproximadamente 15 minutos está alinhada ao perfil de corridas curtas em Manhattan identificado nas etapas anteriores.

b. A correlação entre a duração e o valor da corrida é baixa (0,17). Isso indica que o tempo de viagem exerce menos influência no preço final do que a distância, que apresentou uma correlação de 0,95.

c. Este resultado é coerente com o sistema de tarifas dos Yellow Cabs de Nova York, onde a distância é o fator predominante e o tempo possui um peso secundário (aplicável principalmente em situações de trânsito intenso).

Um ponto problemático: a duração máxima de 1.438 minutos (dividido por 60, é quase 24 horas) é um outlier terrível e pode ser qualquer coisa; um erro de registro, uma corrida não finalizada corretamente. E esse registro permaneceu mesmo após a limpeza da Silver; esse dado precisaria ser avaliado e, por assim dizer, eliminado.

#### Conclusão:
As corridas apresentam, em média, curta duração (cerca de 15 minutos). A duração possui pouca relação com o valor cobrado, permanecendo a distância percorrida como o fator dominante.